<a href="https://colab.research.google.com/github/Betty-jyw/MMA3001-Occupancy-Prediction/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Occupancy Data Exploration

## Objective

The purpose of this notebook is to explore and audit the occupancy sensor dataset before developing the future occupancy prediction model.

The initial prediction objective of the project is to use historical occupancy information to predict whether a selected building space will be occupied 30 minutes into the future.

Before model development, this notebook will investigate:

- dataset structure and available variables
- dataset size and time range
- missing and duplicate records
- occupancy-status distribution
- headcount distribution
- available devices and floor spaces
- sampling frequency and temporal consistency
- suitability of individual building spaces for occupancy prediction

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

data_path = "/content/drive/MyDrive/MMA3001_Project/data"

files = os.listdir(data_path)

for file in files:
    print(file)

Sensor ID and Locations.xlsx
5EnvSensor_MayToDec2024_180kRows.csv
5occupancySensor_MayToDec2024_9MRows.csv


In [5]:
env_file = data_path + "/5EnvSensor_MayToDec2024_180kRows.csv"

env_df = pd.read_csv(env_file)

print("Dataset shape:", env_df.shape)
print("\nColumn names:")
print(env_df.columns.tolist())

env_df.head()

Dataset shape: (179447, 8)

Column names:
['id', 'sensorid', 'devicetype', 'jsondata', 'status', 'processing_errors', 'createdate', 'processdate']


,id,sensorid,devicetype,jsondata,status,processing_errors,createdate,processdate
0,2,6012002000241,env,"[{""values"": [{""time"": 1702273965, ""value"": 4.1...",Processed,NaN,2024-02-29 03:11:58.210358+11,2024-02-29 03:11:58.210358+11
1,4,6012002000241,env,"[{""values"": [{""time"": 1702273965, ""value"": 4.1...",Processed,NaN,2024-02-29 17:09:08.833822+11,2024-02-29 17:09:08.833822+11
2,5,6012002000326,env,"[{""values"": [{""time"": 1702270857, ""value"": 4.1...",Processed,NaN,2024-02-29 17:09:12.444245+11,2024-02-29 17:09:12.444245+11
3,16,6012002000777,env,"[{""values"": [{""time"": 1683326565, ""value"": 4.1...",Processed,NaN,2024-02-29 17:10:24.443407+11,2024-02-29 17:10:24.443407+11
4,24,6012002000869,env,"[{""values"": [{""time"": 1709186696, ""value"": 4.1...",Processed,NaN,2024-02-29 17:11:11.348578+11,2024-02-29 17:11:11.348578+11


check one jsondata

In [6]:
print(env_df.loc[0, "jsondata"])

[{"values": [{"time": 1702273965, "value": 4.1875}], "variable": {"id": 66, "name": "Battery", "unit": "V", "source": 0, "structure": 32834}}, {"values": [{"time": 1702273965, "value": 603}], "variable": {"id": 67, "name": "Carbon dioxide", "unit": "ppm", "source": 0, "structure": 32835}}, {"values": [{"time": 1702273965, "value": 66}], "variable": {"id": 72, "name": "Humidity", "unit": "HR%", "source": 0, "structure": 32840}}, {"values": [{"time": 1702273965, "value": 6}], "variable": {"id": 70, "name": "Formaldehyde", "unit": "µg/m3", "source": 0, "structure": 32847}}, {"values": [{"time": 1702273965, "value": 1011}], "variable": {"id": 80, "name": "Pressure", "unit": "mb", "source": 0, "structure": 32848}}, {"values": [{"time": 1702273965, "value": 23.8}], "variable": {"id": 84, "name": "Temperature", "unit": "°C", "source": 0, "structure": 32852}}, {"values": [{"time": 1702273965, "value": 14}], "variable": {"id": 99, "name": "Light Volatile Organic Compounds", "unit": "ppb", "sour

check are all data the same

In [7]:
import json

variable_names = set()

for json_string in env_df["jsondata"].dropna():
    try:
        records = json.loads(json_string)

        for record in records:
            variable_names.add(record["variable"]["name"])

    except (json.JSONDecodeError, KeyError, TypeError):
        pass

print("Number of environmental variables:", len(variable_names))

for name in sorted(variable_names):
    print(name)

Number of environmental variables: 11
Battery
Carbon dioxide
Formaldehyde
Humidity
Light Volatile Organic Compounds
Particulate matter 1
Particulate matter 10
Particulate matter 2.5
Particulate matter 4
Pressure
Temperature


## Environmental Sensor Data

The environmental sensor dataset contains 11 measured variables, including carbon dioxide, humidity, temperature, pressure, formaldehyde, volatile organic compounds, and particulate matter measurements.

Before using these measurements as model features, the temporal coverage and sensor availability are investigated.

In [8]:
# Convert the creation timestamp to datetime
env_df["createdate"] = pd.to_datetime(env_df["createdate"], errors="coerce")

print("Number of rows:", len(env_df))
print("Number of unique sensors:", env_df["sensorid"].nunique())

print("\nTime range:")
print("Start:", env_df["createdate"].min())
print("End:", env_df["createdate"].max())

print("\nRecords per sensor:")
print(env_df["sensorid"].value_counts().head(10))

Number of rows: 179447
Number of unique sensors: 5

Time range:
Start: 2024-02-29 03:11:58.210358+11:00
End: 2026-04-22 23:20:33.199878+10:00

Records per sensor:
sensorid
6012002000326    58909
6012002000241    58908
6012002000869    29341
6012002000227    21363
6012002000777    10926
Name: count, dtype: int64


/tmp/ipykernel_960/3638478017.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  env_df["createdate"] = pd.to_datetime(env_df["createdate"], errors="coerce")


## Occupancy Sensor Data

The occupancy dataset is substantially larger than the environmental dataset. An initial sample is therefore inspected before loading and processing the complete dataset.

The initial inspection focuses on the dataset structure, available variables, data types, and occupancy records.

In [9]:
occupancy_file = data_path + "/5occupancySensor_MayToDec2024_9MRows.csv"

# Read only the first 10,000 rows for initial inspection
occ_sample = pd.read_csv(occupancy_file, nrows=10000)

print("Sample shape:", occ_sample.shape)

print("\nColumn names:")
print(occ_sample.columns.tolist())

print("\nData types:")
print(occ_sample.dtypes)

occ_sample.head()

Sample shape: (10000, 8)

Column names:
['id', 'deviceid', 'floorspaceid', 'occupancystatus', 'headcount', 'collecteddate', 'occupancystatuschangedate', 'previousoccupancystatus']

Data types:
id                            int64
deviceid                     object
floorspaceid                 object
occupancystatus              object
headcount                     int64
collecteddate                object
occupancystatuschangedate    object
previousoccupancystatus      object
dtype: object


,id,deviceid,floorspaceid,occupancystatus,headcount,collecteddate,occupancystatuschangedate,previousoccupancystatus
0,136043,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,CurrentlyOccupied,1,2023-11-24 00:36:52+11,2023-11-24 00:36:52+11,NotOccupied
1,136048,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,RecentlyOccupied,0,2023-11-24 00:37:32+11,2023-11-24 00:37:32+11,CurrentlyOccupied
2,136050,56923747-b390-4f2a-8930-808429132266,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,CurrentlyOccupied,1,2023-11-24 00:37:32+11,2023-11-24 00:37:32+11,NotOccupied
3,136052,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,NotOccupied,0,2023-11-24 00:37:52+11,2023-11-24 00:37:52+11,RecentlyOccupied
4,136053,56923747-b390-4f2a-8930-808429132266,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,RecentlyOccupied,0,2023-11-24 00:37:52+11,2023-11-24 00:37:52+11,CurrentlyOccupied


## Full Occupancy Dataset Audit

Because the occupancy dataset contains approximately nine million records, the complete file is audited using chunk-based processing rather than loading the entire dataset into memory at once.

This step investigates the total number of records, temporal coverage, number of occupancy devices, and number of floor spaces.

In [10]:
chunk_size = 500000

total_rows = 0
unique_devices = set()
unique_spaces = set()

min_date = None
max_date = None

for chunk in pd.read_csv(
    occupancy_file,
    usecols=["deviceid", "floorspaceid", "collecteddate"],
    chunksize=chunk_size
):
    total_rows += len(chunk)

    unique_devices.update(chunk["deviceid"].dropna().unique())
    unique_spaces.update(chunk["floorspaceid"].dropna().unique())

    dates = pd.to_datetime(
        chunk["collecteddate"],
        errors="coerce",
        utc=True
    )

    chunk_min = dates.min()
    chunk_max = dates.max()

    if pd.notna(chunk_min):
        if min_date is None or chunk_min < min_date:
            min_date = chunk_min

    if pd.notna(chunk_max):
        if max_date is None or chunk_max > max_date:
            max_date = chunk_max

print("Total rows:", total_rows)
print("Unique occupancy devices:", len(unique_devices))
print("Unique floor spaces:", len(unique_spaces))
print("Start date:", min_date)
print("End date:", max_date)

Total rows: 9091732
Unique occupancy devices: 1
Unique floor spaces: 5
Start date: 2023-11-23 13:36:52+00:00
End date: 2026-04-22 13:02:38+00:00


## Sensor and Location Mapping

The sensor-location mapping file is inspected to determine whether the occupancy and environmental sensor records can be associated with identifiable building spaces.

In [11]:
mapping_file = data_path + "/Sensor ID and Locations.xlsx"

excel_file = pd.ExcelFile(mapping_file)

print("Sheet names:")
print(excel_file.sheet_names)

Sheet names:
['ENV', 'OCC (Incorrect Data - SpaceFloo']


checking env mapping （incorrect）

In [12]:
env_mapping = pd.read_excel(mapping_file, sheet_name="ENV")

print("Shape:", env_mapping.shape)

print("\nColumn names:")
print(env_mapping.columns.tolist())

env_mapping.head(10)

Shape: (78, 6)

Column names:
['Building', 'Floor/Room', 'Area', 'Physical Label', 'Sensor', 'ID']


,Building,Floor/Room,Area,Physical Label,Sensor,ID
0,60,1-PS01,Corridor,ES1104,Env,6012003000486
1,69,1-PS02,Breakout,ES2113,Env,6012003000882
2,60,1.40,Board Room,ES1102,Env,6012003000462
3,60,1.41,Open Office,ES1110,Env,6012003000424
4,60,1.41A,Focus Room 1,ES1101,Env,6012003000448
5,60,1.42,Startup Workshop,ES1103,Env,6012003000479
6,60,1.43,Startup,ES1105,Env,6012003000493
7,60,1.45,Meeting Room,ES1106,Env,6012003000509
8,60,1.47,Corridor,ES1107,Env,6012003000516
9,60,1.49,Workshop,ES1108,Env,6012003000530


environmental csv sensorid (==) Excel ENV ID

In [13]:
env_sensor_ids = [
    6012002000326,
    6012002000241,
    6012002000869,
    6012002000227,
    6012002000777
]

matched_env = env_mapping[
    env_mapping["ID"].isin(env_sensor_ids)
]

print("Number of matched sensors:", len(matched_env))

matched_env

Number of matched sensors: 1


,Building,Floor/Room,Area,Physical Label,Sensor,ID
60,60,G.38,Compactus_upper,EMS107,Env,6012002000869


In [14]:
occ_sheet = "OCC (Incorrect Data - SpaceFloo"

occ_mapping = pd.read_excel(
    mapping_file,
    sheet_name=occ_sheet
)

print("Shape:", occ_mapping.shape)

print("\nColumn names:")
print(occ_mapping.columns.tolist())

occ_mapping.head(10)

Shape: (89, 7)

Column names:
['Building', 'Floor/Room', 'Area', 'ID', 'Note', 'FloorId', 'SpaceId']


,Building,Floor/Room,Area,ID,Note,FloorId,SpaceId
0,60,1-PS01,NaN,8034-2812-ae89,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,fe553808-899c-43a3-96f7-f444f41f3046
1,60,1-PS01,NaN,8034-2812-c28f,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,bd7e25c4-beb5-4d17-9426-859cacfd0f03
2,60,1-PS01,NaN,8034-2812-bfc5,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,67235649-ade6-43d7-a216-180d6dd8e3c1
3,60,1-PS01,NaN,8034-2812-cc64,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,7ac9618b-5096-41a1-85d9-2db3541075e6
4,60,1.38,NaN,8034-2812-ecc1,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,56d7400d-7933-4e4e-9ecc-e5da4c8f991d
5,60,1.39,NaN,missing,Missing Id,94fda9ec-e483-460e-96ce-5767c791c1c1,2387067e-5349-422a-a4f5-08d93ef52a6e
6,60,1.40,NaN,8034-285a-73d4,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,f1bbb258-eb6b-4711-9523-5799bf87e53b
7,60,1.41,NaN,8034-2812-e179,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,b8de6aba-795f-4ebc-9f6c-fc7073dd2dc7
8,60,1.41A,NaN,8034-2812-bfb7,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,02ffc7a7-0dce-4791-a86b-062723514932
9,60,1.41B,NaN,8034-2812-c01f,NaN,94fda9ec-e483-460e-96ce-5767c791c1c1,27bdd98c-e1da-485b-a4df-17ec067ed09a


In [15]:
print("Occupancy floor-space IDs:")

for space in sorted(unique_spaces):
    print(space)

Occupancy floor-space IDs:
0f3f37f8-0335-41e0-ae87-a1d677dfc7d7
227f1068-9e41-469f-bef0-275f27a0bdba
3e5a14df-4b14-4ea5-bb9d-46577c6dc9c8
941f0352-bb69-47d0-a9c5-ac29eb4f7319
cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b


In [16]:
matched_occ = occ_mapping[
    occ_mapping["SpaceId"].isin(unique_spaces)
]

print("Number of matched spaces:", len(matched_occ))

matched_occ[
    ["Building", "Floor/Room", "Area", "ID", "Note", "FloorId", "SpaceId"]
]

Number of matched spaces: 4


,Building,Floor/Room,Area,ID,Note,FloorId,SpaceId
60,60,G.20,NaN,8034-2812-bfed,NaN,54803c37-793b-47ed-8f25-df460ed01d48,cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b
61,60,G.20,NaN,8034-285a-c064,NaN,54803c37-793b-47ed-8f25-df460ed01d48,cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b
64,60,G.25,NaN,8034-285a-72f0,2.4m (Keenan Lab),54803c37-793b-47ed-8f25-df460ed01d48,227f1068-9e41-469f-bef0-275f27a0bdba
65,60,G.25,NaN,8034-2812-6ebd,2.4m (Keenan Lab),54803c37-793b-47ed-8f25-df460ed01d48,227f1068-9e41-469f-bef0-275f27a0bdba


In [17]:
print("Matched rows:", len(matched_occ))
print("Unique matched spaces:", matched_occ["SpaceId"].nunique())

print("\nMatched unique Space IDs:")
print(matched_occ["SpaceId"].unique())

unmatched_spaces = set(unique_spaces) - set(matched_occ["SpaceId"])

print("\nUnmatched occupancy Space IDs:")
for space in sorted(unmatched_spaces):
    print(space)

Matched rows: 4
Unique matched spaces: 2

Matched unique Space IDs:
['cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b'
 '227f1068-9e41-469f-bef0-275f27a0bdba']

Unmatched occupancy Space IDs:
0f3f37f8-0335-41e0-ae87-a1d677dfc7d7
3e5a14df-4b14-4ea5-bb9d-46577c6dc9c8
941f0352-bb69-47d0-a9c5-ac29eb4f7319


## Occupancy Distribution by Floor Space

The five floor spaces are compared to determine their data availability and occupancy characteristics. This analysis will support the selection of a suitable space for subsequent occupancy prediction.

In [18]:
from collections import Counter, defaultdict

space_counts = Counter()
status_counts = defaultdict(Counter)
headcount_stats = defaultdict(list)

for chunk in pd.read_csv(
    occupancy_file,
    usecols=["floorspaceid", "occupancystatus", "headcount"],
    chunksize=500000
):
    for space, count in chunk["floorspaceid"].value_counts().items():
        space_counts[space] += count

    grouped = chunk.groupby(
        ["floorspaceid", "occupancystatus"]
    ).size()

    for (space, status), count in grouped.items():
        status_counts[space][status] += count

print("Records per floor space:\n")

for space, count in space_counts.most_common():
    print(space)
    print("Total records:", count)
    print("Occupancy status:", dict(status_counts[space]))
    print()

Records per floor space:

0f3f37f8-0335-41e0-ae87-a1d677dfc7d7
Total records: 2169461
Occupancy status: {'CurrentlyOccupied': 1934394, 'NotOccupied': 87938, 'RecentlyOccupied': 147129}

941f0352-bb69-47d0-a9c5-ac29eb4f7319
Total records: 1975423
Occupancy status: {'CurrentlyOccupied': 1821607, 'NotOccupied': 61462, 'RecentlyOccupied': 92354}

cca1ac3e-f81f-4ebb-aa2b-8b0bf3e9e53b
Total records: 1894871
Occupancy status: {'CurrentlyOccupied': 1829341, 'NotOccupied': 25139, 'RecentlyOccupied': 40391}

3e5a14df-4b14-4ea5-bb9d-46577c6dc9c8
Total records: 1551583
Occupancy status: {'CurrentlyOccupied': 1509463, 'NotOccupied': 14918, 'RecentlyOccupied': 27202}

227f1068-9e41-469f-bef0-275f27a0bdba
Total records: 1500394
Occupancy status: {'CurrentlyOccupied': 1311089, 'NotOccupied': 76248, 'RecentlyOccupied': 113057}



## Temporal Sampling Characteristics

The occupancy dataset contains a very large number of records. Therefore, the temporal spacing between consecutive observations is investigated to determine whether records represent regularly sampled measurements or frequent event updates.

In [19]:
selected_space = "0f3f37f8-0335-41e0-ae87-a1d677dfc7d7"

space_sample = pd.read_csv(
    occupancy_file,
    usecols=[
        "floorspaceid",
        "occupancystatus",
        "headcount",
        "collecteddate"
    ]
)

space_sample = space_sample[
    space_sample["floorspaceid"] == selected_space
].copy()

space_sample["collecteddate"] = pd.to_datetime(
    space_sample["collecteddate"],
    errors="coerce",
    utc=True
)

space_sample = space_sample.sort_values("collecteddate")

space_sample["time_difference"] = (
    space_sample["collecteddate"].diff().dt.total_seconds()
)

print("Selected space:", selected_space)
print("Number of records:", len(space_sample))

print("\nTime range:")
print("Start:", space_sample["collecteddate"].min())
print("End:", space_sample["collecteddate"].max())

print("\nTime difference between consecutive records (seconds):")
print(space_sample["time_difference"].describe())

space_sample.head(10)

Selected space: 0f3f37f8-0335-41e0-ae87-a1d677dfc7d7
Number of records: 2169461

Time range:
Start: 2023-11-23 13:37:32+00:00
End: 2026-04-22 13:02:18+00:00

Time difference between consecutive records (seconds):
count    2.169460e+06
mean     3.508536e+01
std      1.107428e+03
min      2.000000e+00
25%      2.000000e+00
50%      4.000000e+00
75%      1.800000e+01
max      1.345420e+06
Name: time_difference, dtype: float64


,floorspaceid,occupancystatus,headcount,collecteddate,time_difference
2,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,CurrentlyOccupied,1,2023-11-23 13:37:32+00:00,NaN
4,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,RecentlyOccupied,0,2023-11-23 13:37:52+00:00,20.0
5,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,NotOccupied,0,2023-11-23 13:38:32+00:00,40.0
8,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,CurrentlyOccupied,1,2023-11-23 13:41:32+00:00,180.0
10,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,RecentlyOccupied,0,2023-11-23 13:42:12+00:00,40.0
11,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,NotOccupied,0,2023-11-23 13:42:32+00:00,20.0
12,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,CurrentlyOccupied,1,2023-11-23 16:45:32+00:00,10980.0
13,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,RecentlyOccupied,0,2023-11-23 16:45:52+00:00,20.0
14,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,NotOccupied,0,2023-11-23 16:46:32+00:00,40.0
15,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,CurrentlyOccupied,1,2023-11-23 18:02:52+00:00,4580.0


In [20]:
space_sample["previous_row_status"] = (
    space_sample["occupancystatus"].shift(1)
)

space_sample["status_changed"] = (
    space_sample["occupancystatus"]
    != space_sample["previous_row_status"]
)

print("Total records:", len(space_sample))

print(
    "Status-change records:",
    space_sample["status_changed"].sum()
)

print(
    "Percentage of records with status change:",
    space_sample["status_changed"].mean() * 100
)

print("\nOccupancy status distribution:")
print(
    space_sample["occupancystatus"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Total records: 2169461
Status-change records: 382199
Percentage of records with status change: 17.617233036224206

Occupancy status distribution:
occupancystatus
CurrentlyOccupied    89.16
RecentlyOccupied      6.78
NotOccupied           4.05
Name: proportion, dtype: float64


In [21]:
headcount_summary = (
    space_sample
    .groupby("occupancystatus")["headcount"]
    .agg(["count", "mean", "median", "min", "max"])
)

print(headcount_summary)

print("\nHeadcount distribution by occupancy status:")

for status in space_sample["occupancystatus"].dropna().unique():
    print(f"\n{status}")
    print(
        space_sample.loc[
            space_sample["occupancystatus"] == status,
            "headcount"
        ].value_counts().head(10)
    )

                     count      mean  median  min  max
occupancystatus                                       
CurrentlyOccupied  1934394  2.566583     2.0    1   40
NotOccupied          87938  0.000000     0.0    0    0
RecentlyOccupied    147129  0.000000     0.0    0    0

Headcount distribution by occupancy status:

CurrentlyOccupied
headcount
2     628840
1     606616
3     338883
4     169065
5      82702
6      41339
7      21543
8      11540
9       6681
10      4081
Name: count, dtype: int64

RecentlyOccupied
headcount
0    147129
Name: count, dtype: int64

NotOccupied
headcount
0    87938
Name: count, dtype: int64


## Binary Occupancy Definition

For the prediction task, occupancy is converted into a binary variable representing whether the space is actively occupied.

- `CurrentlyOccupied` is defined as **Occupied (1)**.
- `RecentlyOccupied` is defined as **Not actively occupied (0)**.
- `NotOccupied` is defined as **Not occupied (0)**.

This definition is supported by the headcount data. All `CurrentlyOccupied` observations have a headcount of at least one, whereas all `RecentlyOccupied` and `NotOccupied` observations have a headcount of zero.

The prediction target is therefore defined as whether the selected space will be actively occupied 30 minutes into the future.

In [22]:
space_sample["occupied"] = (
    space_sample["occupancystatus"] == "CurrentlyOccupied"
).astype(int)

space_sample[
    ["collecteddate", "occupancystatus", "headcount", "occupied"]
].head(10)

,collecteddate,occupancystatus,headcount,occupied
2,2023-11-23 13:37:32+00:00,CurrentlyOccupied,1,1
4,2023-11-23 13:37:52+00:00,RecentlyOccupied,0,0
5,2023-11-23 13:38:32+00:00,NotOccupied,0,0
8,2023-11-23 13:41:32+00:00,CurrentlyOccupied,1,1
10,2023-11-23 13:42:12+00:00,RecentlyOccupied,0,0
11,2023-11-23 13:42:32+00:00,NotOccupied,0,0
12,2023-11-23 16:45:32+00:00,CurrentlyOccupied,1,1
13,2023-11-23 16:45:52+00:00,RecentlyOccupied,0,0
14,2023-11-23 16:46:32+00:00,NotOccupied,0,0
15,2023-11-23 18:02:52+00:00,CurrentlyOccupied,1,1


## Five-Minute Resampling

The raw occupancy observations are irregularly spaced and frequently recorded within short time intervals. To create a consistent time-series representation, the selected space is resampled onto a five-minute time grid.

For each five-minute timestamp, the most recently observed occupancy state is used to represent the known state of the space. This produces a regular time series suitable for constructing a 30-minute-ahead prediction target.

In [23]:
occupancy_5min = (
    space_sample
    .set_index("collecteddate")[["occupied"]]
    .resample("5min")
    .last()
)

print("Number of 5-minute intervals:", len(occupancy_5min))

print("\nMissing intervals before filling:")
print(occupancy_5min["occupied"].isna().sum())

occupancy_5min.head(20)

Number of 5-minute intervals: 253722

Missing intervals before filling:
128922


,occupied
collecteddate,
2023-11-23 13:35:00+00:00,0.0
2023-11-23 13:40:00+00:00,0.0
2023-11-23 13:45:00+00:00,NaN
2023-11-23 13:50:00+00:00,NaN
2023-11-23 13:55:00+00:00,NaN
2023-11-23 14:00:00+00:00,NaN
2023-11-23 14:05:00+00:00,NaN
2023-11-23 14:10:00+00:00,NaN
2023-11-23 14:15:00+00:00,NaN


In [24]:
missing = occupancy_5min["occupied"].isna()

gap_group = (missing != missing.shift()).cumsum()

missing_gap_lengths = (
    occupancy_5min[missing]
    .groupby(gap_group[missing])
    .size()
)

missing_gap_minutes = missing_gap_lengths * 5

print("Number of missing gaps:", len(missing_gap_minutes))

print("\nMissing gap duration (minutes):")
print(missing_gap_minutes.describe())

print("\nMost common gap durations:")
print(
    missing_gap_minutes
    .value_counts()
    .sort_index()
    .head(20)
)

Number of missing gaps: 13299

Missing gap duration (minutes):
count    13299.000000
mean        48.470562
std        229.633355
min          5.000000
25%          5.000000
50%         10.000000
75%         35.000000
max      22420.000000
dtype: float64

Most common gap durations:
5      5293
10     1961
15     1128
20      662
25      503
30      361
35      285
40      206
45      176
50      259
55      367
60      138
65      100
70       96
75       89
80       84
85       73
90       55
95       57
100      54
Name: count, dtype: int64


## Missing-Data Treatment

The five-minute time series contains both short and long periods without direct sensor observations. The median missing-gap duration is 10 minutes, while some gaps extend for several days.

To avoid assuming that an occupancy state remains unchanged across long periods without sensor observations, forward filling is limited to a maximum of 10 minutes (two five-minute intervals). Longer gaps are retained as missing values and are excluded where reliable occupancy information is required.

The effect of this threshold can later be examined through sensitivity analysis.

In [25]:
occupancy_5min["occupied_filled"] = (
    occupancy_5min["occupied"].ffill(limit=2)
)

print("Missing intervals before limited filling:")
print(occupancy_5min["occupied"].isna().sum())

print("\nMissing intervals after limited filling:")
print(occupancy_5min["occupied_filled"].isna().sum())

print("\nAvailable intervals after filling:")
print(occupancy_5min["occupied_filled"].notna().sum())

print("\nPercentage available:")
print(
    occupancy_5min["occupied_filled"].notna().mean() * 100
)

Missing intervals before limited filling:
128922

Missing intervals after limited filling:
107617

Available intervals after filling:
146105

Percentage available:
57.584679294661086


## 30-Minute-Ahead Prediction Target

The objective of the model is to predict whether the selected space will be actively occupied 30 minutes into the future.

Because the occupancy time series has been resampled at five-minute intervals, 30 minutes corresponds to six time steps. The future occupancy state is therefore shifted by six intervals to create the prediction target.

Only observations for which both the current occupancy state and the future occupancy target are available are retained for subsequent model development.

In [26]:
occupancy_5min["target_30min"] = (
    occupancy_5min["occupied_filled"].shift(-6)
)

model_data = occupancy_5min[
    occupancy_5min["occupied_filled"].notna()
    & occupancy_5min["target_30min"].notna()
].copy()

print("Total 5-minute intervals:", len(occupancy_5min))
print("Valid samples for 30-minute prediction:", len(model_data))

print("\nTarget distribution:")
print(model_data["target_30min"].value_counts())

print("\nTarget distribution (%):")
print(
    model_data["target_30min"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

model_data[
    ["occupied", "occupied_filled", "target_30min"]
].head(15)

Total 5-minute intervals: 253722
Valid samples for 30-minute prediction: 127428

Target distribution:
target_30min
1.0    77939
0.0    49489
Name: count, dtype: int64

Target distribution (%):
target_30min
1.0    61.16
0.0    38.84
Name: proportion, dtype: float64


,occupied,occupied_filled,target_30min
collecteddate,,,
2023-11-23 19:15:00+00:00,0.0,0.0,0.0
2023-11-23 20:50:00+00:00,0.0,0.0,0.0
2023-11-23 20:55:00+00:00,0.0,0.0,1.0
2023-11-23 21:00:00+00:00,0.0,0.0,1.0
2023-11-23 21:05:00+00:00,0.0,0.0,0.0
2023-11-23 21:10:00+00:00,NaN,0.0,0.0
2023-11-23 21:15:00+00:00,0.0,0.0,0.0
2023-11-23 21:20:00+00:00,0.0,0.0,1.0
2023-11-23 21:25:00+00:00,1.0,1.0,1.0


In [27]:
check_data = occupancy_5min[
    ["occupied_filled", "target_30min"]
].copy()

check_data["future_time"] = check_data.index + pd.Timedelta(minutes=30)

print("Example current and future timestamps:\n")

for current_time, row in check_data.dropna().head(10).iterrows():
    print(
        "Current:",
        current_time,
        "| Future:",
        row["future_time"],
        "| Current occupancy:",
        int(row["occupied_filled"]),
        "| Target:",
        int(row["target_30min"])
    )

Example current and future timestamps:

Current: 2023-11-23 19:15:00+00:00 | Future: 2023-11-23 19:45:00+00:00 | Current occupancy: 0 | Target: 0
Current: 2023-11-23 20:50:00+00:00 | Future: 2023-11-23 21:20:00+00:00 | Current occupancy: 0 | Target: 0
Current: 2023-11-23 20:55:00+00:00 | Future: 2023-11-23 21:25:00+00:00 | Current occupancy: 0 | Target: 1
Current: 2023-11-23 21:00:00+00:00 | Future: 2023-11-23 21:30:00+00:00 | Current occupancy: 0 | Target: 1
Current: 2023-11-23 21:05:00+00:00 | Future: 2023-11-23 21:35:00+00:00 | Current occupancy: 0 | Target: 0
Current: 2023-11-23 21:10:00+00:00 | Future: 2023-11-23 21:40:00+00:00 | Current occupancy: 0 | Target: 0
Current: 2023-11-23 21:15:00+00:00 | Future: 2023-11-23 21:45:00+00:00 | Current occupancy: 0 | Target: 0
Current: 2023-11-23 21:20:00+00:00 | Future: 2023-11-23 21:50:00+00:00 | Current occupancy: 0 | Target: 1
Current: 2023-11-23 21:25:00+00:00 | Future: 2023-11-23 21:55:00+00:00 | Current occupancy: 1 | Target: 1
Curren

## Data Exploration Summary

The occupancy dataset was investigated to determine its suitability for 30-minute-ahead occupancy prediction.

The selected building space contained more than two million raw occupancy records. The observations were irregularly spaced and highly frequent, with a median interval of approximately four seconds between consecutive records.

The original occupancy states were converted into a binary representation. `CurrentlyOccupied` was classified as occupied (1), while `RecentlyOccupied` and `NotOccupied` were classified as not actively occupied (0). This definition was supported by the headcount data, where `CurrentlyOccupied` observations had a headcount of at least one and the other two states had a headcount of zero.

To create a regular time series, the data were resampled at five-minute intervals. Short missing periods were forward-filled for a maximum of 10 minutes, while longer gaps remained missing to avoid making unsupported assumptions about occupancy.

A 30-minute-ahead prediction target was then created using six five-minute time steps. After removing samples without reliable current or future occupancy information, 127,428 valid observations remained. Approximately 61.16% of the prediction targets were occupied and 38.84% were not occupied.

The processed time series is therefore suitable for subsequent feature engineering and machine-learning model development.

In [28]:
print("Final dataset shape:", occupancy_5min.shape)

print("\nColumns:")
print(occupancy_5min.columns.tolist())

occupancy_5min.head(10)

Final dataset shape: (253722, 3)

Columns:
['occupied', 'occupied_filled', 'target_30min']


,occupied,occupied_filled,target_30min
collecteddate,,,
2023-11-23 13:35:00+00:00,0.0,0.0,NaN
2023-11-23 13:40:00+00:00,0.0,0.0,NaN
2023-11-23 13:45:00+00:00,NaN,0.0,NaN
2023-11-23 13:50:00+00:00,NaN,0.0,NaN
2023-11-23 13:55:00+00:00,NaN,NaN,NaN
2023-11-23 14:00:00+00:00,NaN,NaN,NaN
2023-11-23 14:05:00+00:00,NaN,NaN,NaN
2023-11-23 14:10:00+00:00,NaN,NaN,NaN
2023-11-23 14:15:00+00:00,NaN,NaN,NaN


In [29]:
model_data = occupancy_5min[
    occupancy_5min["occupied_filled"].notna() &
    occupancy_5min["target_30min"].notna()
].copy()

print("Number of valid modelling samples:", len(model_data))

print("\nColumns:")
print(model_data.columns.tolist())

print("\nMissing values:")
print(model_data.isna().sum())

model_data.head(10)

Number of valid modelling samples: 127428

Columns:
['occupied', 'occupied_filled', 'target_30min']

Missing values:
occupied           12546
occupied_filled        0
target_30min           0
dtype: int64


,occupied,occupied_filled,target_30min
collecteddate,,,
2023-11-23 19:15:00+00:00,0.0,0.0,0.0
2023-11-23 20:50:00+00:00,0.0,0.0,0.0
2023-11-23 20:55:00+00:00,0.0,0.0,1.0
2023-11-23 21:00:00+00:00,0.0,0.0,1.0
2023-11-23 21:05:00+00:00,0.0,0.0,0.0
2023-11-23 21:10:00+00:00,NaN,0.0,0.0
2023-11-23 21:15:00+00:00,0.0,0.0,0.0
2023-11-23 21:20:00+00:00,0.0,0.0,1.0
2023-11-23 21:25:00+00:00,1.0,1.0,1.0


In [30]:
processed_data = model_data[
    ["occupied_filled", "target_30min"]
].copy()

processed_data = processed_data.rename(
    columns={"occupied_filled": "current_occupancy"}
)

processed_data.head(10)

,current_occupancy,target_30min
collecteddate,,
2023-11-23 19:15:00+00:00,0.0,0.0
2023-11-23 20:50:00+00:00,0.0,0.0
2023-11-23 20:55:00+00:00,0.0,1.0
2023-11-23 21:00:00+00:00,0.0,1.0
2023-11-23 21:05:00+00:00,0.0,0.0
2023-11-23 21:10:00+00:00,0.0,0.0
2023-11-23 21:15:00+00:00,0.0,0.0
2023-11-23 21:20:00+00:00,0.0,1.0
2023-11-23 21:25:00+00:00,1.0,1.0


In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import os

drive_path = "/content/drive/MyDrive"

print("Files and folders in MyDrive:")
print(os.listdir(drive_path))

Files and folders in MyDrive:
['IMMI Acknowledgement Letter.pdf', 'Untitled presentation (1).gslides', 'Untitled document (16).gdoc', 'Copy of MCD4540 Milestone 1 Template for Students (1).gdoc', 'Untitled document (15).gdoc', 'Copy of MCD4540 Milestone 1 Template for Students.gdoc', 'week 5 table.gdoc', 'MCD4540 Circuit Diagram Template.drawio', 'Seven_segment_display_diagram 的副本.drawio', 'lab (1).drawio', 'Book report.gdoc', 'MCD4540 Week 8 Workshop.gdoc', 'MCD4540 Circuit Diagram Template 的副本 (2).drawio', 'MCD4540 Circuit Diagram Template 的副本 (1).drawio', 'MCD4520 Project 1 Report Group 1.gdoc', 'MCD4540 Circuit Diagram Template 的副本.drawio', 'Copy of Milestone 3 Feature Selection 2024 T2 - MAKE A COPY (1).gsheet', 'lab.drawio', 'Copy of Milestone 3 Feature Selection 2024 T2 - MAKE A COPY.gsheet', 'Copy of Copy of Milestone 3 Feature Selection 2024 T2 - MAKE A COPY.gsheet', 'Untitled document (14).gdoc', 'Copy of Generative AI Record of Interaction Template.docx', 'Untitled document 

In [33]:
project_path = "/content/drive/MyDrive/MMA3001_Project"

print("Files in MMA3001_Project:")
print(os.listdir(project_path))

Files in MMA3001_Project:
['data']


In [34]:
data_path = "/content/drive/MyDrive/MMA3001_Project/data"

print("Files in data:")
print(os.listdir(data_path))

Files in data:
['Sensor ID and Locations.xlsx', '5EnvSensor_MayToDec2024_180kRows.csv', '5occupancySensor_MayToDec2024_9MRows.csv']


In [35]:
processed_path = "/content/drive/MyDrive/MMA3001_Project/data/processed"

# Create the folder if it does not already exist
os.makedirs(processed_path, exist_ok=True)

# Save the processed dataset
output_file = os.path.join(
    processed_path,
    "occupancy_processed.csv"
)

processed_data.to_csv(output_file, index=True)

print("Processed dataset saved to:")
print(output_file)

print("\nSaved shape:")
print(processed_data.shape)

Processed dataset saved to:
/content/drive/MyDrive/MMA3001_Project/data/processed/occupancy_processed.csv

Saved shape:
(127428, 2)
